IMPORTING LIBRARIES

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


True

SETTING ENVIRONMENT VARIABLES

In [2]:
groq_api_key = os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
huggingface_api_key = os.environ["HUGGINGFACE_API_KEY"] = os.getenv("HUGGINGFACE_API_KEY")

INITIALIZING LLM

In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(api_key=groq_api_key, model="groq/compound-mini")
llm

ChatGroq(output_version=None, profile={}, client=<groq.resources.chat.completions.Completions object at 0x7fa77bb19010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7fa77bb19d30>, model_name='groq/compound-mini', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

LOADING WEBPAGE CONTENT

In [4]:
from langchain_docling import DoclingLoader

loader = DoclingLoader(file_path="https://docs.langchain.com/oss/python/langchain/quickstart/")

documents = loader.load()
documents


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (518 > 512). Running this sequence through the model will result in indexing errors


[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/quickstart/', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/27', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/28', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/29', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/30', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}], 'headings': ['Get started'], 'origin': {'mimetype': 'text/html', 'binary_hash': 6756311161503709715, 'filename': 'quickstart'}}}, page_content='Get started\n- [Install](/oss/python/langchain/install)\n- [Quickstart](/oss/python/langchain/quickstart)\n- [

SPLITTING DOCUMENT INTO CHUNKS

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

documents = text_splitter.split_documents(documents)
documents

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/quickstart/', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/27', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/28', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/29', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/30', 'parent': {'$ref': '#/groups/5'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}], 'headings': ['Get started'], 'origin': {'mimetype': 'text/html', 'binary_hash': 6756311161503709715, 'filename': 'quickstart'}}}, page_content='Get started\n- [Install](/oss/python/langchain/install)\n- [Quickstart](/oss/python/langchain/quickstart)\n- [

STORING EMBEDDINGS IN VECTORSTORE

In [6]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(documents, embeddings)
vector_store.save_local("./faiss_db")


/tmp/ipykernel_151012/3669629388.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

CREATING RETRIEVER FOR VECTORSTORE

In [7]:
retriever = vector_store.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fa736df0440>, search_kwargs={})

DEFINING PROMPT TEMPLATE

In [8]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a helpful AI assistant that helps people find information "
    "about Langchain Quickstart. Use the following pieces of "
    "context to answer the question at the end. If you don't know the answer, "
    "just say you don't know, don't try to make up an answer. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("user", "{input}"),])

CREATING CHAIN

In [10]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [11]:
response = qa_chain.invoke("What are the requirements?")


In [12]:
from IPython.display import Markdown, display
display(Markdown(response))


**Requirements to get started with the Groq API quick‑start**

| Category | What you need |
|----------|----------------|
| **Account** | A Groq account so you can generate an **API key** (via the Groq console). |
| **API key handling** | Ability to set environment variables (e.g., `export GROQ_API_KEY=…`) – this is the recommended way to keep the key out of your code. |
| **Programming language/runtime** | • **Python** ≥ 3.8 (or any supported language you prefer).<br>• **Node.js** if you want to use the JavaScript AI SDK. |
| **Package manager** | • `pip` for Python (`pip install groq`).<br>• `pnpm`/`npm`/`yarn` for JavaScript (`pnpm add ai @ai-sdk/groq`). |
| **Network access** | Outbound HTTPS access to `https://api.groq.com` (the Groq endpoint). |
| **Optional tools** | • `curl` (for quick command‑line testing).<br>• A terminal/command‑prompt where you can set env vars and run scripts.<br>• (If using the AI SDK) a modern browser or Node environment. |
| **Optional but helpful** | • Groq Playground (web UI) for interactive testing.<br>• Access to the Groq developer community or documentation for further guidance. |

In short, you just need a Groq API key, a supported runtime (Python or Node), the corresponding SDK/library installed, and the ability to set an environment variable for the key. Once those are in place you can run the sample code and start making chat completions.

ADDING CONVERSATION HISTORY

In [13]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

contextualize_system_prompt = (
    "Given the conversation history and a follow up question,"
    "rephrase the follow up question to be a standalone question."
    "If the follow up question is not related to the history, just say you don't"
    "have context to answer. Don't try to make up an answer.")

# Answers questions using retrieved documents and chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])

# Rephrases follow-up questions to standalone questions
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])

contextualize_q_chain = contextualize_q_prompt | llm | StrOutputParser()


In [14]:
def get_context(input_dict):
    if input_dict.get("chat_history"):
        rephrased = contextualize_q_chain.invoke(input_dict)
    else:
        rephrased = input_dict["input"]
    return format_docs(retriever.invoke(rephrased))


In [15]:
qa_chain = (
    RunnablePassthrough.assign(context=get_context)
    | qa_prompt
    | llm
    | StrOutputParser()
)


In [18]:
import time
from langchain_core.messages import AIMessage, HumanMessage

chat_history = []
question = "What are the requirements?"
response = qa_chain.invoke({
    "input": question,
    "chat_history": chat_history
})

chat_history.extend([HumanMessage(content=question), AIMessage(content=response)])

time.sleep(10)  # wait for TPM window to partially reset

question2 = "Are there any code examples?"
response2 = qa_chain.invoke({
    "input": question2,
    "chat_history": chat_history
})
display(Markdown(response2))


Below are a few of the most common code snippets that appear in the **LangChain Quickstart** documentation.  
You can copy‑paste them into a fresh Python file (or a Jupyter notebook) after you have installed the required packages and set your API keys.

---

## 1️⃣ Minimal “Hello‑World” LLM chain  

```python
# 1️⃣ Install the core libraries first
# pip install langchain langchain-community   # (or `pip install deepagents tavily-python` for the full quickstart)

from langchain.llms import OpenAI   # works with any provider that has an OpenAI‑compatible API
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Prompt template – you can change the text however you like
prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question in one short sentence:\n{question}"
)

# Build the chain (LLM + prompt)
chain = LLMChain(llm=OpenAI(model="gpt-4o-mini"), prompt=prompt)

# Run it
response = chain.invoke({"question": "What is the capital of France?"})
print(response["text"])
# → "Paris."
```

*What it shows:*  
* How to import an LLM wrapper (`OpenAI`), create a prompt template, and run a single‑step chain.

---

## 2️⃣ Conversational Retrieval‑Augmented Generation (RAG)  

```python
# pip install langchain-community[vectorstores]  # e.g., for Chroma, FAISS, Pinecone, etc.

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OpenAIEmbeddings
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.llms import OpenAI

# 1️⃣ Load / create a vector store (here we use a local Chroma DB)
embeddings = OpenAIEmbeddings()
vectorstore = Chroma(persist_directory="my_chroma_db", embedding_function=embeddings)

# 2️⃣ Set up a conversational chain that can retrieve relevant docs
retrieval_chain = ConversationalRetrievalChain.from_llm(
    llm=OpenAI(model="gpt-4o-mini"),
    retriever=vectorstore.as_retriever(),
    memory=ConversationBufferMemory(memory_key="chat_history")
)

# 3️⃣ Talk to it
print(retrieval_chain.invoke({"question": "What did we discuss about LangChain last week?"})["answer"])
```

*What it shows:*  
* Using a vector store for document retrieval.  
* Adding memory so the model can keep context across turns.

---

## 3️⃣ **Real‑world agent** – an agent that can browse the web, run Python, and use a calculator  

```python
# pip install langchain-community[tools] tavily-python

from langchain.agents import AgentExecutor, create_openai_functions_agent
from langchain.llms import OpenAI
from langchain.tools import TavilySearchResults, PythonREPLTool, Calculator

# 1️⃣ Define the tools you want the agent to have
tools = [
    TavilySearchResults(max_results=5),   # web‑search tool (requires TAVILY_API_KEY)
    PythonREPLTool(),                     # lets the agent run arbitrary Python code
    Calculator()                          # simple arithmetic
]

# 2️⃣ Create the LLM (any OpenAI‑compatible model works)
llm = OpenAI(model="gpt-4o-mini", temperature=0)

# 3️⃣ Build the agent that knows how to call the tools
agent = create_openai_functions_agent(llm, tools)

# 4️⃣ Wrap it in an executor (handles the loop of tool calls)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 5️⃣ Ask a multi‑step question
question = """
Find the current population of Tokyo from a reliable source, then calculate what
percentage of the world’s population that represents (use the latest world‑population
estimate you can find). Show your calculations.
"""

result = agent_executor.invoke({"input": question})
print(result["output"])
```

*What it shows:*  
* How to give an agent **tool access** (web search, Python REPL, calculator).  
* The `verbose=True` flag prints each intermediate step, which is useful when you later enable **LangSmith** tracing.

---

## 4️⃣ Enabling **LangSmith** tracing (optional but highly recommended)

```bash
export LANGSMITH_TRACING="true"
export LANGSMITH_API_KEY="your‑langsmith‑api‑key"
```

Then, in any of the scripts above, just add:

```python
from langchain import tracing

tracing.start_tracing()
# ... run your chain / agent as usual ...
tracing.end_tracing()
```

When you run the script, the full request/response graph will appear in your LangSmith dashboard, letting you debug prompts, see token usage, and compare runs.

---

### How to run the snippets

1. **Create a fresh virtual environment** (highly recommended).  
   ```bash
   python -m venv .venv
   source .venv/bin/activate   # Windows: .venv\Scripts\activate
   ```
2. **Install the needed packages** (pick the ones you need).  
   ```bash
   pip install langchain langchain-community openai
   # plus any extras shown above, e.g.:
   pip install tavily-python chromadb
   ```
3. **Set your API keys** (OpenAI, Tavily, etc.) as environment variables.  
   ```bash
   export OPENAI_API_KEY="sk-..."
   export TAVILY_API_KEY="tavily-..."
   ```
4. **Copy a snippet into a file** (`quickstart_example.py`) and run it:  
   ```bash
   python quickstart_example.py
   ```

---

### Quick reference cheat‑sheet

| Goal | One‑liner import & call |
|------|------------------------|
| **Simple LLM completion** | `print(OpenAI(model="gpt-4o-mini").invoke("Write a haiku about rain."))` |
| **Chain with prompt** | `LLMChain(llm=OpenAI(), prompt=PromptTemplate.from_template("{question}")).invoke({"question":"Explain recursion"})` |
| **Agent with web search** | See snippet #3 (Tavily + PythonREPL + Calculator). |
| **RAG with vector store** | See snippet #2 (Chroma + ConversationalRetrievalChain). |
| **Trace with LangSmith** | Set `LANGSMITH_TRACING=true` and wrap calls with `tracing.start_tracing()` / `tracing.end_tracing()`. |

These examples cover the core patterns the LangChain quickstart walks you through: a plain LLM call, a prompt‑driven chain, a retrieval‑augmented conversation, and a tool‑enabled autonomous agent. Feel free to mix‑and‑match the pieces to build the exact workflow you need!